In [ ]:
# --- Sistema y Google Drive ---
import os
import glob
from google.colab import drive

# --- Procesamiento de Datos y Evaluación ---
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix

# --- Procesamiento de Imágenes y Visión Artificial ---
import cv2
from skimage import io, data
from skimage.color import rgb2gray
from tensorflow.keras.preprocessing.image import load_img, img_to_array

# --- Visualización ---
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Image, display

# --- Deep Learning (TensorFlow y Keras) ---
import tensorflow as tf
from tensorflow.keras import layers, models, backend as K
from tensorflow.keras.utils import plot_model, model_to_dot
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.layers import (
    Dense, Dropout, Flatten, Conv2D, 
    MaxPool2D, BatchNormalization, Conv2DTranspose, 
    Input, Activation, concatenate
)

In [ ]:
# Definición de directorios
dataset_dir = "/content/drive/MyDrive/New_U_Net/RGB"

train_img_dir = f"{dataset_dir}/Set_A_img"
train_mask_dir = f"{dataset_dir}/Set_A_mask"
test_img_dir  = f"{dataset_dir}/Set_B_img"
test_mask_dir = f"{dataset_dir}/Set_B_mask"

# Obtención y ordenamiento de rutas de archivos
train_img_paths = sorted(glob.glob(train_img_dir + "/*.jpg"))
train_mask_paths = sorted(glob.glob(train_mask_dir + "/*.png"))

test_img_paths = sorted(glob.glob(test_img_dir + "/*.jpg"))
test_mask_paths = sorted(glob.glob(test_mask_dir + "/*.png"))

# Visualización de conteo de archivos
print("Train images:", len(train_img_paths))
print("Train masks:", len(train_mask_paths))
print("Test images:", len(test_img_paths))
print("Test masks:", len(test_mask_paths))

In [ ]:
# Dimensiones de entrada 
IMG_HEIGHT = 128
IMG_WIDTH = 128
BATCH_SIZE = 4

In [ ]:
# Normalización de imagen-máscara
def load_image(img_path, mask_path):
    # Procesamiento de la imagen RGB
    img = tf.io.read_file(img_path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, (IMG_HEIGHT, IMG_WIDTH))
    img = tf.cast(img, tf.float32) / 255.0  # Normalización [0, 1]

    # Procesamiento de la máscara (Ground Truth)
    mask = tf.io.read_file(mask_path)
    mask = tf.image.decode_png(mask, channels=1)
    mask = tf.image.resize(mask, (IMG_HEIGHT, IMG_WIDTH))
    mask = tf.cast(mask > 127, tf.float32)  # Binarización a 0 y 1

    return img, mask

In [ ]:
# Indexacion y creacion 
train_img_paths = sorted(glob.glob(train_img_dir + "/*.jpg"))
train_mask_paths = sorted(glob.glob(train_mask_dir + "/*.png"))
test_img_paths = sorted(glob.glob(test_img_dir + "/*.jpg"))
test_mask_paths = sorted(glob.glob(test_mask_dir + "/*.png"))

print(f"Imágenes de entrenamiento encontradas: {len(train_img_paths)}")
print(f"Imágenes de prueba encontradas: {len(test_img_paths)}")

#  Creación del Dataset principal de entrenamiento
train_dataset = tf.data.Dataset.from_tensor_slices((train_img_paths, train_mask_paths))
train_dataset = train_dataset.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)

#  División en Entrenamiento (80%) y Validación (20%)
train_size = int(0.8 * len(train_img_paths))
val_size = len(train_img_paths) - train_size

train_ds = train_dataset.take(train_size).shuffle(100).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
val_ds   = train_dataset.skip(train_size).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

#  Creación del Dataset de prueba (Test)
test_ds = tf.data.Dataset.from_tensor_slices((test_img_paths, test_mask_paths))
test_ds = test_ds.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
test_ds = test_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

print("-" * 30)
print(f"Dataset de Entrenamiento (train_ds) listo: {train_size} muestras.")
print(f"Dataset de Validación (val_ds) listo:    {val_size} muestras.")
print(f"Dataset de Prueba (test_ds) listo:      {len(test_img_paths)} muestras.")

In [ ]:
# IoU
def iou_metric(y_true, y_pred, smooth=1e-6):
    y_true_f = K.flatten(y_true)
    y_pred_f = K.flatten(y_pred)
    intersection = K.sum(y_true_f * y_pred_f)
    union = K.sum(y_true_f) + K.sum(y_pred_f) - intersection
    return (intersection + smooth) / (union + smooth)

# Dice Coefficient
def dice_coef(y_true, y_pred, smooth=1e-6):
    y_true_f = K.flatten(y_true)
    y_pred_f = K.flatten(y_pred)
    intersection = K.sum(y_true_f * y_pred_f)
    return (2. * intersection + smooth) / (K.sum(y_true_f) + K.sum(y_pred_f) + smooth)

# Sensibilidad (Recall)
def sensitivity(y_true, y_pred):
    y_true_f = tf.reshape(y_true, [-1])
    y_pred_f = tf.reshape(tf.round(y_pred), [-1])
    TP = tf.reduce_sum(y_true_f * y_pred_f)
    FN = tf.reduce_sum(y_true_f * (1 - y_pred_f))
    return TP / (TP + FN + tf.keras.backend.epsilon())

# Especificidad 
def specificity(y_true, y_pred):
    y_true_f = tf.reshape(y_true, [-1])
    y_pred_f = tf.reshape(tf.round(y_pred), [-1])
    TN = tf.reduce_sum((1 - y_true_f) * (1 - y_pred_f))
    FP = tf.reduce_sum((1 - y_true_f) * y_pred_f)
    return TN / (TN + FP + tf.keras.backend.epsilon())

In [ ]:
def unet_model(input_size=(128, 128, 3)):
    inputs = layers.Input(input_size)

    # data augmentation
    data_augmentation = tf.keras.Sequential([
        layers.RandomFlip("horizontal_and_vertical"),
        layers.RandomRotation(0.2),
        layers.RandomZoom(0.2),
        layers.RandomContrast(0.2),
    ])
    x = data_augmentation(inputs)

    # Encoder
    c1 = layers.Conv2D(64, (3,3), padding='same')(inputs)
    c1 = layers.BatchNormalization()(c1)
    c1 = layers.Activation('relu')(c1)
    c1 = layers.Conv2D(64, (3,3), padding='same')(c1)
    c1 = layers.BatchNormalization()(c1)
    c1 = layers.Activation('relu')(c1)
    p1 = layers.MaxPooling2D((2,2))(c1)

    c2 = layers.Conv2D(128, (3,3), padding='same')(p1)
    c2 = layers.BatchNormalization()(c2)
    c2 = layers.Activation('relu')(c2)
    c2 = layers.Conv2D(128, (3,3), padding='same')(c2)
    c2 = layers.BatchNormalization()(c2)
    c2 = layers.Activation('relu')(c2)
    p2 = layers.MaxPooling2D((2,2))(c2)

    c3 = layers.Conv2D(256, (3,3), padding='same')(p2)
    c3 = layers.BatchNormalization()(c3)
    c3 = layers.Activation('relu')(c3)
    c3 = layers.Conv2D(256, (3,3), padding='same')(c3)
    c3 = layers.BatchNormalization()(c3)
    c3 = layers.Activation('relu')(c3)
    p3 = layers.MaxPooling2D((2,2))(c3)

    c4 = layers.Conv2D(512, (3,3), padding='same')(p3)
    c4 = layers.BatchNormalization()(c4)
    c4 = layers.Activation('relu')(c4)
    c4 = layers.Conv2D(512, (3,3), padding='same')(c4)
    c4 = layers.BatchNormalization()(c4)
    c4 = layers.Activation('relu')(c4)
    p4 = layers.MaxPooling2D((2,2))(c4)

    # Bridge
    c5 = layers.Conv2D(1024, (3,3), padding='same')(p4)
    c5 = layers.BatchNormalization()(c5)
    c5 = layers.Activation('relu')(c5)
    c5 = layers.Conv2D(1024, (3,3), padding='same')(c5)
    c5 = layers.BatchNormalization()(c5)
    c5 = layers.Activation('relu')(c5)

    # Decoder
    u6 = layers.Conv2DTranspose(512, (2,2), strides=(2,2), padding='same')(c5)
    u6 = layers.concatenate([u6, c4])
    c6 = layers.Conv2D(512, (3,3), padding='same')(u6)
    c6 = layers.BatchNormalization()(c6)
    c6 = layers.Activation('relu')(c6)
    c6 = layers.Conv2D(512, (3,3), padding='same')(c6)
    c6 = layers.BatchNormalization()(c6)
    c6 = layers.Activation('relu')(c6)

    u7 = layers.Conv2DTranspose(256, (2,2), strides=(2,2), padding='same')(c6)
    u7 = layers.concatenate([u7, c3])
    c7 = layers.Conv2D(256, (3,3), padding='same')(u7)
    c7 = layers.BatchNormalization()(c7)
    c7 = layers.Activation('relu')(c7)
    c7 = layers.Conv2D(256, (3,3), padding='same')(c7)
    c7 = layers.BatchNormalization()(c7)
    c7 = layers.Activation('relu')(c7)

    u8 = layers.Conv2DTranspose(128, (2,2), strides=(2,2), padding='same')(c7)
    u8 = layers.concatenate([u8, c2])
    c8 = layers.Conv2D(128, (3,3), padding='same')(u8)
    c8 = layers.BatchNormalization()(c8)
    c8 = layers.Activation('relu')(c8)
    c8 = layers.Conv2D(128, (3,3), padding='same')(c8)
    c8 = layers.BatchNormalization()(c8)
    c8 = layers.Activation('relu')(c8)

    u9 = layers.Conv2DTranspose(64, (2,2), strides=(2,2), padding='same')(c8)
    u9 = layers.concatenate([u9, c1])
    c9 = layers.Conv2D(64, (3,3), padding='same')(u9)
    c9 = layers.BatchNormalization()(c9)
    c9 = layers.Activation('relu')(c9)
    c9 = layers.Conv2D(64, (3,3), padding='same')(c9)
    c9 = layers.BatchNormalization()(c9)
    c9 = layers.Activation('relu')(c9)

    outputs = layers.Conv2D(1, (1,1), activation='sigmoid')(c9)

    model = models.Model(inputs=[inputs], outputs=[outputs])
    return model



model = unet_model()


model.compile(optimizer="adam",
              loss="binary_crossentropy",
              metrics=["accuracy", iou_metric, dice_coef, sensitivity, specificity])


In [ ]:
# Configuracion Callbacks
earlystop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss', 
    patience=5, 
    restore_best_weights=True
)

# Inicio del proceso de entrenamiento
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=[earlystop]
)

In [ ]:
results = model.evaluate(test_ds)
print("Test results:", results)